# 09 · Modelos predictivos e interpretabilidad

**Objetivo.** Extra de portafolio, separado del análisis inferencial: comparar algoritmos con validación agrupada por conglomerado y contrastar SHAP con los OR.

| | |
|---|---|
| **Entradas** | `data.parquet` |
| **Salidas** | `tabla_28` a `tabla_30`, `fig_13`, `fig_14`, `model_card.md` |

> Proyecto: *Modelamiento epidemiológico de la depresión en la población peruana de 15 y
> más años (ENDES 2025)* · Epidemiología 2026-2 · UNMSM


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "src"))
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "scripts"))

import numpy as np
import pandas as pd
from depresion import config as cfg, io as dio, viz
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
viz.aplicar_tema()
print("Proyecto:", cfg.RAIZ.name)

> **Esta sección responde otra pregunta.** El análisis epidemiológico pregunta
> «¿cuál es el efecto de X?»; aquí se pregunta «¿se puede ordenar el riesgo?». Son
> objetivos distintos y no se sustituyen.

In [ ]:
%run ../scripts/analisis_06_ml.py

In [ ]:
pd.read_csv(cfg.TABLAS / "tabla_28_comparacion_modelos_ml.csv")[
    ["modelo", "auc_roc", "auc_pr", "sensibilidad", "especificidad", "f1", "exactitud"]]

## El contraste que importa

¿El modelo aprende lo mismo que dice la epidemiología? Comparar el ranking de SHAP con el
de los OR ajustados responde a eso, y la discrepancia es más informativa que la
coincidencia.

In [ ]:
pd.read_csv(cfg.TABLAS / "tabla_30_shap_vs_or.csv")

## Conclusiones

- AUC-ROC entre 0,65 y 0,68 según el algoritmo; AUC-PR de 0,11 a 0,12 frente a 0,062 del
  azar, es decir **1,9 veces mejor**.
- La validación se agrupa por conglomerado: repartir personas del mismo conglomerado entre
  entrenamiento y prueba habría inflado el rendimiento.
- **No se reporta la exactitud como métrica principal.** Con 6,2 % de eventos, responder
  «nadie tiene depresión» acierta el 93,8 %.
- **SHAP y los OR discrepan, y ahí está lo interesante.** El consumo problemático de
  alcohol es el **primero por OR (6,07)** y el **decimoquinto por SHAP**: afecta a solo 304
  personas, así que un efecto grande sobre muy pocos aporta poco a la predicción media.
  El sexo es lo contrario: efecto moderado sobre media población, y domina el SHAP.
- La lección: **un OR grande no implica una variable útil para predecir**, y una variable
  importante para predecir no es necesariamente la más relevante para intervenir.
